# OlmoEarth v1.1-Base, frozen encoder (768-d embeddings)

This notebook follows `src/models/olmoearth.py` and shows how the wrapper adapts benchmark Sentinel-2 series into the OlmoEarth sample format.

Key properties:
- Uses Sentinel-2 only.
- Expects 12 OlmoEarth channels; benchmark-missing B01 and B09 are filled and masked.
- Builds images, masks, and timestamps before calling the pretrained encoder.
- Pools the encoder output into one 768-dimensional embedding per parcel.

## What OlmoEarth expects as input

| Tensor | Shape | Meaning |
| --- | --- | --- |
| `images` | `(N, 1, 1, T, 12)` | Sentinel-2 series in OlmoEarth channel order |
| `masks` | `(N, 1, 1, T, 1)` | v1.1 all-band token masks |
| `timestamps` | `(N, T, 3)` | day, zero-indexed month, year for every observation |
| output embedding | `(N, 768)` | pooled frozen representation |

OlmoEarth has its own package dependency. The conversion cell below reports that dependency clearly if it is not installed in the active environment.

In [ ]:
import os
import sys
import importlib.util
from pathlib import Path
from types import SimpleNamespace

import numpy as np

REPO = Path.cwd()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))


def load(name, rel):
    spec = importlib.util.spec_from_file_location(name, REPO / rel)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


In [ ]:
olmoearth = load('olmoearth_mod', 'src/models/olmoearth.py')

print('OlmoEarth bands:', olmoearth.OLMOEARTH_S2_BANDS)
print('embedding dim:', olmoearth.OLMOEARTH_EMBEDDING_DIM)
print('HF repo:', olmoearth.OLMOEARTH_HF_REPO)


In [ ]:
rng = np.random.default_rng(7)
N, T = 4, 18

s2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'NDVI']
s1_bands = ['VV', 'VH']
climate_bands = ['temperature', 'precipitation', 'elevation', 'slope']

bench = SimpleNamespace(
    name='synthetic',
    s2=(rng.random((N, T, len(s2_bands))) * 5000).astype('float32'),
    s1=rng.normal(-12, 3, size=(N, T, len(s1_bands))).astype('float32'),
    climate=rng.normal(0, 1, size=(N, T, len(climate_bands))).astype('float32'),
    s2_mask=np.ones((N, T), dtype='float32'),
    s1_mask=np.ones((N, T), dtype='float32'),
    climate_mask=np.ones((N, T), dtype='float32'),
    doy=np.tile(np.linspace(15, 350, T), (N, 1)).astype('float32'),
    latlon=np.repeat(np.array([[11.0, 79.0]], dtype='float32'), N, axis=0),
    years=np.full(N, 2021, dtype='int64'),
    s2_bands=s2_bands,
    s1_bands=s1_bands,
    climate_bands=climate_bands,
)

print('s2', bench.s2.shape, bench.s2_bands)
print('s1', bench.s1.shape, bench.s1_bands)
print('climate', bench.climate.shape, bench.climate_bands)
print('doy', bench.doy.shape, 'latlon', bench.latlon.shape)


## Wrapper mapping

`_bench_to_olmoearth` is the model-specific conversion used internally by `encode`. It relies on `olmoearth_pretrain` for the sample datatypes and normalization helpers.

In [ ]:
enc = olmoearth.OlmoEarthEncoder(device='cpu')

try:
    images, masks, timestamps = enc._bench_to_olmoearth(bench)
    print('images', images.shape, images.dtype)
    print('masks', masks.shape, masks.dtype)
    print('timestamps', timestamps.shape, timestamps.dtype)
except ModuleNotFoundError as exc:
    print('Skipping conversion because olmoearth_pretrain is not installed:', exc)
except Exception as exc:
    print('Skipping conversion because the wrapper raised:', type(exc).__name__, exc)


## Forward pass -> embedding

The real forward pass requires the OlmoEarth weights. The shared project environment includes the v1.1 package and downloads the public Base weights on first use.

In [ ]:
RUN_REAL_FORWARD = False

enc = olmoearth.OlmoEarthEncoder(device='cpu')

if not RUN_REAL_FORWARD:
    print('Set RUN_REAL_FORWARD = True to run the frozen encoder.')
else:
    emb = enc.encode(bench)
    print('embedding', emb.shape)
    print('first row, first 8 dims', np.round(emb[0, :8], 4))
